In [0]:
%pip install openai

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
# NOTEBOOK 4 — LLM Evaluation Pipeline with Hallucination Detection

# This notebook:
# 1. Uses the Champion model to score loan applications
# 2. For rejected applicants, generates a plain-English explanation using an LLM
# 3. Evaluates each explanation for:
#    - Factual grounding (does it reference real applicant data?)
#    - Hallucination (does it cite a regulation not in the RBI/SEBI list?)
# 4. Saves all eval scores to a governed Delta table
# =============================================================================

# CELL 1 — Install + Import
%pip install openai pandas numpy mlflow transformers torch lightgbm

import pandas as pd
import numpy as np
import mlflow
import mlflow.pyfunc
import re
import json
from datetime import datetime
from pyspark.sql import SparkSession
from mlflow.tracking import MlflowClient
import warnings
warnings.filterwarnings("ignore")

spark = SparkSession.builder.getOrCreate()
client = MlflowClient()
mlflow.set_experiment("/Users/rajath2010rrp@gmail.com/hackbricks_credit_risk")

print("Imports done.")

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Imports done.


- This is the "curated list of real regulations" mentioned in the problem.
- In real life, you'd parse actual RBI circulars. For the hackathon,
- we create a reference table with real-sounding (and some real) RBI guidelines.
- This gets stored as a Delta table — the hallucination checker reads from it.

In [0]:
# -----------------------------------------------------------------------------
# CELL 2 — Build the RBI/SEBI Reference List (Ground Truth)
#
# This is the "curated list of real regulations" mentioned in the problem.
# In real life, you'd parse actual RBI circulars. For the hackathon,
# we create a reference table with real-sounding (and some real) RBI guidelines.
#
# This gets stored as a Delta table — the hallucination checker reads from it.
# -----------------------------------------------------------------------------



# These are modeled after real RBI guidelines (simplified for the hackathon)
rbi_sebi_regulations = [
    {
        "regulation_id":   "RBI/2023-24/73",
        "category":        "Credit Assessment",
        "description":     "Guidelines on credit risk assessment for retail borrowers. Debt-to-income ratio must not exceed 50% for personal loans.",
        "keywords":        ["debt-to-income", "DTI", "income ratio", "personal loan"]
    },
    {
        "regulation_id":   "RBI/2022-23/117",
        "category":        "Credit Bureau",
        "description":     "Mandatory credit bureau check for all loan applications above Rs 10,000. Minimum CIBIL score of 650 required.",
        "keywords":        ["credit score", "CIBIL", "credit bureau", "credit check"]
    },
    {
        "regulation_id":   "RBI/2021-22/158",
        "category":        "Income Verification",
        "description":     "Lenders must verify income through bank statements or Form 16. Self-declared income not accepted for loans above Rs 5 lakh.",
        "keywords":        ["income verification", "bank statement", "Form 16", "salary"]
    },
    {
        "regulation_id":   "RBI/2023-24/41",
        "category":        "Employment",
        "description":     "Minimum 6 months of employment stability required for salaried borrowers. Minimum 2 years of business continuity for self-employed.",
        "keywords":        ["employment", "job stability", "salaried", "self-employed", "business continuity"]
    },
    {
        "regulation_id":   "SEBI/HO/2022/134",
        "category":        "Collateral",
        "description":     "Loan-to-value ratio for secured loans must not exceed 75% for residential property and 60% for commercial property.",
        "keywords":        ["loan-to-value", "LTV", "collateral", "property", "secured loan"]
    },
    {
        "regulation_id":   "RBI/2023-24/98",
        "category":        "Existing Debt",
        "description":     "Total EMI obligations must not exceed 40% of net monthly income. Existing loan defaults in last 12 months disqualify applicant.",
        "keywords":        ["EMI", "monthly obligation", "existing debt", "default history"]
    },
    {
        "regulation_id":   "RBI/2020-21/89",
        "category":        "Age",
        "description":     "Loan tenure must end before borrower's 70th birthday. Minimum age for loan applicants is 21 years.",
        "keywords":        ["age", "tenure", "retirement", "minimum age"]
    },
    {
        "regulation_id":   "RBI/2022-23/201",
        "category":        "Loan Amount",
        "description":     "Maximum loan amount for personal loans without collateral is 10x of monthly net income.",
        "keywords":        ["loan amount", "income multiplier", "maximum loan", "unsecured"]
    }
]

# Convert to DataFrame and save as Delta table
reg_df = pd.DataFrame(rbi_sebi_regulations)
spark.createDataFrame(reg_df) \
    .write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("hackbricks.rbi_sebi_regulations")

print(f"Regulation reference table created with {len(reg_df)} entries.")
print("Saved to: hackbricks.rbi_sebi_regulations")
display(reg_df[['regulation_id', 'category', 'description']])

# Create a set of valid regulation IDs for fast lookup
VALID_REGULATION_IDS = set(reg_df['regulation_id'].tolist())
print(f"\nValid regulation IDs: {VALID_REGULATION_IDS}")


Regulation reference table created with 8 entries.
Saved to: hackbricks.rbi_sebi_regulations


regulation_id,category,description
RBI/2023-24/73,Credit Assessment,Guidelines on credit risk assessment for retail borrowers. Debt-to-income ratio must not exceed 50% for personal loans.
RBI/2022-23/117,Credit Bureau,"Mandatory credit bureau check for all loan applications above Rs 10,000. Minimum CIBIL score of 650 required."
RBI/2021-22/158,Income Verification,Lenders must verify income through bank statements or Form 16. Self-declared income not accepted for loans above Rs 5 lakh.
RBI/2023-24/41,Employment,Minimum 6 months of employment stability required for salaried borrowers. Minimum 2 years of business continuity for self-employed.
SEBI/HO/2022/134,Collateral,Loan-to-value ratio for secured loans must not exceed 75% for residential property and 60% for commercial property.
RBI/2023-24/98,Existing Debt,Total EMI obligations must not exceed 40% of net monthly income. Existing loan defaults in last 12 months disqualify applicant.
RBI/2020-21/89,Age,Loan tenure must end before borrower's 70th birthday. Minimum age for loan applicants is 21 years.
RBI/2022-23/201,Loan Amount,Maximum loan amount for personal loans without collateral is 10x of monthly net income.



Valid regulation IDs: {'RBI/2023-24/98', 'RBI/2022-23/201', 'RBI/2023-24/73', 'SEBI/HO/2022/134', 'RBI/2023-24/41', 'RBI/2022-23/117', 'RBI/2021-22/158', 'RBI/2020-21/89'}


In [0]:
from openai import OpenAI

llm_client = OpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url=f"{DATABRICKS_HOST}/serving-endpoints"
)

# Test it works before running on all applicants
test = llm_client.chat.completions.create(
    model="databricks-meta-llama-3-3-70b-instruct",
    messages=[{"role": "user", "content": "Say hello in one sentence."}],
    max_tokens=20
)
print("LLM test:", test.choices[0].message.content)

LLM test: Hello, it's nice to meet you and I'm here to help with any questions or topics you


In [0]:
# CELL 3 — Load Champion model and score applicants

print("Loading Champion model from MLflow registry...")

champion_model = mlflow.pyfunc.load_model("models:/credit-risk-champion@Champion")
print("Champion model loaded.")

# Load some applicants to score
app_df = pd.read_csv("/Volumes/workspace/default/hackbricks/application_train.csv")

# Quick cleaning (same as Notebook 1)
from sklearn.preprocessing import LabelEncoder
threshold = 0.6
cols_to_drop = [col for col in app_df.columns if app_df[col].isnull().mean() > threshold]
app_df_clean = app_df.drop(columns=cols_to_drop)

X_all = app_df_clean.drop(columns=['TARGET', 'SK_ID_CURR'], errors='ignore')
categorical_cols = X_all.select_dtypes(include=['object']).columns.tolist()
le = LabelEncoder()
for col in categorical_cols:
    X_all[col] = X_all[col].fillna("MISSING")
    X_all[col] = le.fit_transform(X_all[col].astype(str))
X_all = X_all.fillna(X_all.median())

# Score a sample of 100 applicants (keeping it small for speed)
sample_size = 100
sample_idx  = np.random.choice(len(X_all), size=sample_size, replace=False)
X_sample    = X_all.iloc[sample_idx].reset_index(drop=True)
app_sample  = app_df_clean.iloc[sample_idx].reset_index(drop=True)

# Get default probabilities
default_probs = champion_model.predict(X_sample)

# If model returns class labels (0/1), we need predict_proba
# Load as LightGBM directly for probabilities
import mlflow.lightgbm
lgbm_model = mlflow.lightgbm.load_model("models:/credit-risk-champion@Champion")
default_probs = lgbm_model.predict_proba(X_sample)[:, 1]

# Rejection threshold: if default probability > 0.5, loan is rejected
REJECTION_THRESHOLD = float(np.percentile(default_probs, 80))
print(f"Dynamic rejection threshold (80th percentile): {REJECTION_THRESHOLD:.4f}")

rejected_mask = default_probs > REJECTION_THRESHOLD
rejected_apps = app_sample[rejected_mask].copy()
rejected_probs = default_probs[rejected_mask]

print(f"Scored {sample_size} applicants.")
print(f"Rejected: {rejected_mask.sum()}")
print(f"Approved: {(~rejected_mask).sum()}")

rejected_apps['default_probability'] = rejected_probs
rejected_apps = rejected_apps.head(20)
print(f"Processing {len(rejected_apps)} rejection explanations.")
rejected_apps = app_sample[rejected_mask].copy()
rejected_probs = default_probs[rejected_mask]

print(f"\nScored {sample_size} applicants.")
print(f"Rejected (default prob > {REJECTION_THRESHOLD}): {rejected_mask.sum()}")
print(f"Approved: {(~rejected_mask).sum()}")

# For the LLM eval, we work with the rejected ones
rejected_apps['default_probability'] = rejected_probs
rejected_apps = rejected_apps.head(20)  # Take first 20 rejections for the demo
print(f"Processing {len(rejected_apps)} rejection explanations.")

Loading Champion model from MLflow registry...


Champion model loaded.


Dynamic rejection threshold (80th percentile): 0.1106
Scored 100 applicants.
Rejected: 20
Approved: 80
Processing 20 rejection explanations.

Scored 100 applicants.
Rejected (default prob > 0.11061483305328534): 20
Approved: 80
Processing 20 rejection explanations.


In [0]:
# Run this cell to see which LLM endpoints are available
import requests

response = requests.get(
    f"{DATABRICKS_HOST}/api/2.0/serving-endpoints",
    headers={"Authorization": f"Bearer {DATABRICKS_TOKEN}"}
)

endpoints = response.json().get('endpoints', [])
print("Available serving endpoints:")
for ep in endpoints:
    print(f"  - {ep['name']}  (state: {ep.get('state', {}).get('ready', 'unknown')})")

Available serving endpoints:
  - databricks-gpt-5-4  (state: READY)
  - databricks-gpt-5-4-mini  (state: READY)
  - databricks-gpt-5-4-nano  (state: READY)
  - databricks-gpt-5-2  (state: READY)
  - databricks-gpt-oss-120b  (state: READY)
  - databricks-gpt-5-3-codex  (state: READY)
  - databricks-gpt-5-2-codex  (state: READY)
  - databricks-gpt-5-1-codex-max  (state: READY)
  - databricks-gpt-5-1-codex-mini  (state: READY)
  - databricks-gpt-oss-20b  (state: READY)
  - databricks-qwen3-next-80b-a3b-instruct  (state: READY)
  - databricks-llama-4-maverick  (state: READY)
  - databricks-gemma-3-12b  (state: READY)
  - databricks-gte-large-en  (state: READY)
  - databricks-bge-large-en  (state: READY)
  - databricks-gpt-5-1  (state: READY)
  - databricks-meta-llama-3-1-8b-instruct  (state: READY)
  - databricks-meta-llama-3-3-70b-instruct  (state: READY)
  - databricks-qwen3-embedding-0-6b  (state: READY)
  - databricks-meta-llama-3.1-405b-instruct  (state: READY)


In [0]:
# -----------------------------------------------------------------------------
# CELL 4 — Generate Rejection Explanations using LLM
#
# OPTION A (Recommended for hackathon): Use Databricks Model Serving
#   - Databricks has built-in LLM endpoints (DBRX, Llama 3, etc.)
#   - No API key needed if using Databricks-hosted models
#
# OPTION B: Use OpenAI API
#   - Needs an API key
#
# We show OPTION A using Databricks Foundation Model API.
# For each rejected applicant, we build a prompt with their actual data
# and ask the LLM to explain the rejection.
# -----------------------------------------------------------------------------


spark = SparkSession.builder.getOrCreate()

DATABRICKS_HOST  = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
LLM_ENDPOINT = f"{DATABRICKS_HOST}/api/2.0/serving-endpoints/databricks-meta-llama-3-3-70b-instruct/invocations"
# OR use: "databricks-llama-3-1-70b" if DBRX is not available in your workspace


def build_rejection_prompt(applicant_row):

    income        = applicant_row.get('AMT_INCOME_TOTAL', 0) or 0
    credit        = applicant_row.get('AMT_CREDIT', 0) or 0
    annuity       = applicant_row.get('AMT_ANNUITY', 0) or 0
    days_birth    = applicant_row.get('DAYS_BIRTH', 0) or 0
    days_employed = applicant_row.get('DAYS_EMPLOYED', 0) or 0
    default_prob  = applicant_row.get('default_probability', 0) or 0

    age              = int(abs(days_birth) // 365) if days_birth else 0
    employment_years = int(abs(days_employed) // 365) if days_employed < 0 else 0

    income_str  = f"Rs {int(income):,}"  if income  else "N/A"
    credit_str  = f"Rs {int(credit):,}"  if credit  else "N/A"
    annuity_str = f"Rs {int(annuity):,}" if annuity else "N/A"

    try:
        dti     = (annuity / income * 100) if income > 0 else 0
        dti_str = f"{dti:.1f}%"
    except:
        dti_str = "N/A"
        dti     = 0

    prob_str = f"{float(default_prob):.1%}"

    # Tell the LLM clearly why this person is in the rejected pile
    # so it doesn't invent reasons
    if dti > 50:
        primary_reason = f"debt-to-income ratio of {dti_str} exceeds the 50% threshold"
    elif employment_years < 1:
        primary_reason = f"employment duration of {employment_years} years is below minimum stability requirement"
    elif default_prob > 0.15:
        primary_reason = f"overall credit risk score of {prob_str} is above acceptable threshold"
    else:
        primary_reason = f"combination of loan amount {credit_str} and income {income_str} produces unfavorable risk profile"

    valid_reg_list = "\n".join([
        f"  - {r['regulation_id']}: {r['description'][:80]}..."
        for r in rbi_sebi_regulations
    ])

    prompt = f"""You are a loan officer at an Indian bank. Write a professional rejection letter.

APPLICANT DATA:
- Monthly Income: {income_str}
- Loan Amount Requested: {credit_str}
- Monthly EMI: {annuity_str}
- Debt-to-Income Ratio: {dti_str}
- Age: {age} years
- Employment Duration: {employment_years} years
- Default Risk Score: {prob_str}

PRIMARY REJECTION REASON (use this as the main reason — do not invent other reasons):
{primary_reason}

VALID REGULATIONS (ONLY cite IDs from this list):
{valid_reg_list}

INSTRUCTIONS:
1. Write 2-3 sentences based ONLY on the primary rejection reason above
2. Reference the applicant's actual numbers
3. Cite the most relevant regulation ID from the list if it matches the reason
4. Do NOT mention reasons not supported by the data above

Rejection explanation:"""

    return prompt


# REDEFINE call_llm — this overwrites the old version in memory
from openai import OpenAI
import re

llm_client = OpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url=f"{DATABRICKS_HOST}/serving-endpoints"
)

def call_llm(prompt):
    try:
        response = llm_client.chat.completions.create(
            model="databricks-meta-llama-3-3-70b-instruct",
            messages=[
                {
                    "role": "system",
                    "content": "You are a professional loan officer at an Indian bank writing rejection letters. Be concise, factual, and reference the applicant's actual data."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            max_tokens=200,
            temperature=0.3
        )
        return response.choices[0].message.content.strip()

    except Exception as e:
        print(f"  Error: {str(e)}")
        income = re.search(r'Monthly Income: Rs ([\d,]+)', prompt)
        credit = re.search(r'Loan Amount Requested: Rs ([\d,]+)', prompt)
        dti    = re.search(r'Debt-to-Income Ratio: ([\d.]+)%', prompt)
        inc_str = income.group(1) if income else "N/A"
        crd_str = credit.group(1) if credit else "N/A"
        dti_val = float(dti.group(1)) if dti else 0
        reg     = "RBI/2023-24/73" if dti_val > 50 else "RBI/2023-24/98"
        return (
            f"Your loan application for Rs {crd_str} has been rejected. "
            f"Your monthly income of Rs {inc_str} results in an unfavorable "
            f"debt-to-income ratio per {reg}. Please reapply after improving your credit profile."
        )

# Verify the new function works before running the loop
test = call_llm("Write one sentence rejecting a loan due to high debt.")
print("Function test:", test)

# Generate explanations for all rejected applicants
print("Generating LLM explanations for rejected applicants...")
explanations = []

for idx, row in rejected_apps.iterrows():
    prompt = build_rejection_prompt(row)
    explanation = call_llm(prompt)
    explanations.append(explanation)

    if (len(explanations)) % 5 == 0:
        print(f"  Generated {len(explanations)}/{len(rejected_apps)} explanations...")

rejected_apps['llm_explanation'] = explanations
print(f"\nAll {len(explanations)} explanations generated.")

# Show a few examples
print("\n=== Sample explanations ===")
for i, (idx, row) in enumerate(rejected_apps.head(3).iterrows()):
    print(f"\nApplicant {i+1}:")
    print(f"  Default Prob: {row['default_probability']:.1%}")
    print(f"  Explanation: {row['llm_explanation']}")


Function test: We regret to inform you that your loan application has been rejected due to your high debt-to-income ratio of 0.75, which exceeds our acceptable limit of 0.60, as per the credit report and financial statements you submitted.
Generating LLM explanations for rejected applicants...
  Generated 5/20 explanations...
  Generated 10/20 explanations...
  Generated 15/20 explanations...
  Generated 20/20 explanations...

All 20 explanations generated.

=== Sample explanations ===

Applicant 1:
  Default Prob: 11.3%
  Explanation: We regret to inform you that your loan application has been rejected due to your employment duration of 0 years, which falls short of our minimum stability requirement. As per RBI/2023-24/41, a minimum of 6 months of employment stability is mandatory for salaried borrowers, and unfortunately, your current employment status does not meet this criterion. Your loan application of Rs 1,107,981 cannot be approved at this time due to this employment instabilit

In [0]:
# DEBUG CELL — run this alone before the loop
import requests
import json

# Test with a simple hardcoded prompt first
test_response = requests.post(
    LLM_ENDPOINT,
    headers={
        "Authorization": f"Bearer {DATABRICKS_TOKEN}",
        "Content-Type": "application/json"
    },
    json={
        "messages": [
            {"role": "user", "content": "Say hello in one sentence."}
        ],
        "max_tokens": 50,
        "temperature": 0.1
    },
    timeout=60
)

print(f"Status code: {test_response.status_code}")
print(f"Response headers: {dict(test_response.headers)}")
print(f"Response body: {test_response.text[:500]}")

Status code: 404
Response headers: {'date': 'Sat, 11 Apr 2026 17:25:46 GMT', 'content-type': 'application/json', 'x-databricks-org-id': '7474656095211125', 'strict-transport-security': 'max-age=31536000; includeSubDomains; preload', 'x-content-type-options': 'nosniff', 'x-request-id': '56e22527-3468-40a5-8e56-6fe08398ca09', 'content-encoding': 'gzip', 'vary': 'Accept-Encoding', 'server': 'databricks', 'server-timing': 'request_id;dur=0;desc="56e22527-3468-40a5-8e56-6fe08398ca09", client_protocol;dur=0;desc="HTTP/1.1"', 'transfer-encoding': 'chunked'}
Response body: {"error_code":"ENDPOINT_NOT_FOUND","message":"No API found for 'POST /serving-endpoints/databricks-meta-llama-3-3-70b-instruct/invocations'","details":[{"@type":"type.googleapis.com/google.rpc.RequestInfo","request_id":"56e22527-3468-40a5-8e56-6fe08398ca09","serving_data":""}]}


In [0]:
# -----------------------------------------------------------------------------
# CELL 5 — Custom Evaluation Metrics
#
# Now we build the hallucination detection and factual grounding scorers.
# These are custom metrics that mlflow.evaluate() will use.
# -----------------------------------------------------------------------------

def extract_regulation_ids(text):
    """
    Extract all regulation IDs mentioned in the text.
    RBI IDs look like: RBI/2022-23/117
    SEBI IDs look like: SEBI/HO/2022/134
    """
    # Pattern: matches RBI/YYYY-YY/NNN or SEBI/HO/YYYY/NNN
    pattern = r'(?:RBI|SEBI)/[\w\-/]+'
    matches = re.findall(pattern, text)
    return [m.strip('.,;)') for m in matches]


def score_hallucination(explanation, valid_ids):
    """
    Check if the explanation cites any non-existent regulations.

    Returns:
        score: 0.0 = pure hallucination, 1.0 = no hallucinations
        cited_ids: list of regulation IDs mentioned
        fake_ids: list of IDs that don't exist in reference
        real_ids: list of IDs that ARE in reference
    """
    cited_ids = extract_regulation_ids(explanation)

    if not cited_ids:
        # No regulations cited — not a hallucination but also not grounded
        return 1.0, [], [], []

    fake_ids = [rid for rid in cited_ids if rid not in valid_ids]
    real_ids = [rid for rid in cited_ids if rid in valid_ids]

    # Score: what fraction of cited regulations are real?
    score = len(real_ids) / len(cited_ids) if cited_ids else 1.0

    return round(score, 2), cited_ids, fake_ids, real_ids


def score_factual_grounding(explanation, applicant_row):
    """
    Check if the explanation references actual data from the applicant's record.
    We look for numbers that appear in the applicant's data.

    Returns:
        score: 0.0 = pure generic text, 1.0 = highly grounded in real data
    """
    # Extract numbers from the explanation
    numbers_in_explanation = set(re.findall(r'\d[\d,\.]+', explanation))

    # Extract key applicant values (as strings with some formatting variations)
    income = applicant_row.get('AMT_INCOME_TOTAL', 0)
    credit = applicant_row.get('AMT_CREDIT', 0)
    annuity = applicant_row.get('AMT_ANNUITY', 0)
    age = abs(applicant_row.get('DAYS_BIRTH', 0)) // 365

    applicant_values = set()
    for val in [income, credit, annuity, age]:
        if val and val != 0:
            applicant_values.add(str(int(val)))
            applicant_values.add(f"{int(val):,}")
            # Add partial match (first 3 digits) for approximate references
            applicant_values.add(str(int(val))[:4])

    # Count how many applicant values appear in the explanation
    matches = 0
    for exp_num in numbers_in_explanation:
        clean_num = exp_num.replace(',', '')
        for app_val in applicant_values:
            clean_app = app_val.replace(',', '')
            if clean_num == clean_app or clean_num in clean_app:
                matches += 1
                break

    # Also check for keywords that indicate data reference
    grounding_keywords = ['income', 'salary', 'loan amount', 'EMI', 'debt', 'ratio',
                          'age', 'employment', 'years', '%', 'Rs', 'lakh']
    keyword_score = sum(1 for kw in grounding_keywords if kw.lower() in explanation.lower())

    # Combined score: 60% from numbers, 40% from keywords
    number_score   = min(1.0, matches / 2.0)      # 2+ matching numbers = perfect
    keyword_score_normalized = min(1.0, keyword_score / 3.0)  # 3+ keywords = perfect

    final_score = 0.6 * number_score + 0.4 * keyword_score_normalized
    return round(final_score, 2)


print("Custom evaluation functions defined.")

# Quick test
test_explanation_good = "Your loan of Rs 5,00,000 has been rejected because your debt-to-income ratio of 68% exceeds the 50% limit per RBI/2023-24/73. Your monthly income of Rs 32,000 is insufficient to service an EMI of Rs 21,750."
test_explanation_bad  = "Your loan has been rejected per RBI Circular DBR.No.BP.2019 due to general creditworthiness concerns."

halluc_score_good, cited, fake, real = score_hallucination(test_explanation_good, VALID_REGULATION_IDS)
halluc_score_bad,  _,     _,    _    = score_hallucination(test_explanation_bad,  VALID_REGULATION_IDS)

print(f"\nGood explanation hallucination score: {halluc_score_good} (cited: {cited}, fake: {fake})")
print(f"Bad explanation hallucination score:  {halluc_score_bad}")